Preprocess data for recommender setup. 
* Also subsets data according to clinical trial task setup. (and subsets cointexts by that also).
* Merges some sparse columns together into single text col


tf-rs +keras example, 2 tower: expedia:
* https://medium.com/expedia-group-tech/candidate-generation-using-a-two-tower-approach-with-expedia-group-traveler-data-ca6a0dcab83e

keras-rs, tensorflow recommenders ?
* deepmatch, deepctr, ms recc;
* https://github.com/shenweichen/DeepMatch/blob/master/examples/colab_MovieLen1M_SDM.ipynb
* Maybe try LibRecommender (seems useful): https://librecommender.readthedocs.io/en/latest/user_guide/feature_engineering.html
    * libReco doesn't have support for text features!

Pretrained GO embeddings - could use anc2vec or others

STRING (+- protein) embeddings: https://github.com/deweihu96/SPACE
* https://chatgpt.com/share/68a30d02-0fd4-8013-aadc-5f2e475a9c69 
* human‑specific string embedding: `9606.protein.network.embeddings.v12.0.h5`. Contains two datasets – embeddings (N×D array) and proteins (a list of STRING protein identifiers)
    * STRING  alias file: `9606.protein.aliases.v12.0.txt.gz` maps each STRING protein ID to various aliases (including Ensembl gene IDs).
* https://github.com/krishnanlab/PecanPy - fast training
* network features, embeddings: biosnap? STRING, bioGrid
    * https://snap.stanford.edu/biodata/index.html
 
Pretrained text embeddings - (I don't understand the code + it's on different (news) data). 
* https://github.com/ebanalyse/ebnerd-benchmark/blob/main/src/ebrec/models/newsrec/nrms_docvec.py 
* https://github.com/ebanalyse/ebnerd-benchmark/blob/main/examples/quick_start/nrms_ebnerd.ipynb
* https://github.com/ebanalyse/ebnerd-benchmark/blob/17d2f308073389cb7f4bd40c3136e891706ff721/src/ebrec/utils/_articles.py#L21
* https://github.com/recommenders-team/recommenders/blob/main/examples/02_model_content_based_filtering/dkn_deep_dive.ipynb

In [ ]:
import os
import numpy as np
import pandas as pd
import shap
# from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, GroupKFold, StratifiedGroupKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score, accuracy_score, precision_score, recall_score

# import tensorflow as tf, keras

# DATA_DIR = "./opentargets/"
DATA_DIR = "../data/opentargets/"

In [ ]:
SAVE_INTERMEDIATES =False
SAVE_NOVEL_PREDICTION_CANDIDATES = False 

FAST_RUN = False#True
# FAST_RUN = True

RUN_CV = True

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, make_scorer, average_precision_score
from catboost import CatBoostClassifier, Pool, cv

# ------------------------ config ---------------------------------
CFG = dict(
    parquet_path   = "df_sample.parquet",
    target_col     = "label",
    embed_cols     = ["diseaseEmbeddings", "targetEmbeddings"],
    text_cols      = ["description",
                      "ancestors",
                      # "therapeuticAreas",
                      "go",
                      # "synonyms",
                      "functionDescriptions","subcellularLocations",
                      # "pathways", "targetClass",
                      "tractability",
                      # 'dbXRefs',
                     ],
    cat_cols       = ["diseaseId","targetId","biotype"],
    cb_params      = dict(iterations=30 if FAST_RUN else 1000, depth=4 if FAST_RUN else 8,#depth=10 , ~ 2500 iter # for good perf
                          loss_function="Logloss",
                          eval_metric="AUC", # not supported on gpu? 
                          early_stopping_rounds=50, #verbose=False,
                          task_type="GPU", ## runs out of mem with many cols
                          random_seed=42,
                         auto_class_weights = "SqrtBalanced",
                         # dictionaries= ['Word:min_token_occurrence=5']#,'BiGram:gram_order=2'
                          # max_ctr_complexity = 3,
                        ),
    cv_folds       = 5,
    # max_ctr_complexity = 3, # default 4 ? 
)


cat_features       = ["diseaseId","biotype"] # ,"biotype"
# text_features      = ["disease_text","target_text"] # # could maybe use sparse tfidf feats instead +- embedding features
# embed_cols     = ["diseaseEmbeddings", "targetEmbeddings"] # need to keep ID col + merge. ## Note: embeddings = heavy compute! 
embed_cols = None

text_features = ["disease_text","target_text",
                 # 'dbXRefs','therapeuticAreas', ## now merged in text
                 "go","subcellularLocations",
                 "modelPhenotypeClasses","modelPhenotypeId", ## mouse feat
                 "phenotypes","tractability",
                 'ancestors', 
                ]
# text_features      = ["disease_text",] 
# embed_cols     = [ "targetEmbeddings"] # need to keep ID col + merge. ## Note: embeddings = heavy compute! 

In [ ]:
def preprocess_open_targets(df: pd.DataFrame, top_n_tissues: int = 5) -> pd.DataFrame:
    """Preprocess nested DepMap essentiality data into a flat feature table.
    >>> target_essentiality = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target_essentiality'))
    >>> preprocess_open_targets(target_essentiality.head())
    """
    # Determine globally most frequent tissues from context-level data
    all_tissues = []
    for gene_ess_list in df["geneEssentiality"]:
        for gene_ess in gene_ess_list or []:
            for dep in gene_ess.get("depMapEssentiality", []):
                if dep.get("tissueName"):
                    all_tissues.append(dep["tissueName"])
    top_tissues = (
        pd.Series(all_tissues).value_counts().index.tolist()[:top_n_tissues]
    )

    result_records = []
    for _, row in df.iterrows():
        gene_id = row["id"]
        contexts = []
        screens = []

        # flatten contexts and screens; propagate tissueName to each screen
        for gene_ess in row.get("geneEssentiality") or []:
            for dep in gene_ess.get("depMapEssentiality", []):
                contexts.append(dep)
                tissue = dep.get("tissueName")
                for scr in dep.get("screens", []):
                    scr_aug = scr.copy()
                    scr_aug["tissueName"] = tissue  # attach context tissue to screen
                    screens.append(scr_aug)

        if not contexts:
            continue

        # context-level essentiality metrics
        n_contexts = len(contexts)
        n_essential_contexts = sum(1 for c in contexts if c.get("isEssential"))
        essential_ratio = n_essential_contexts / n_contexts

        # screen-level statistics
        screens_df = pd.DataFrame(screens)
        avg_gene_effect = screens_df["geneEffect"].mean()
        std_gene_effect = screens_df["geneEffect"].std(ddof=0)
        min_gene_effect = screens_df["geneEffect"].min()
        max_gene_effect = screens_df["geneEffect"].max()
        avg_expression = screens_df["expression"].mean()
        max_expression = screens_df["expression"].max()
        n_mutations = screens_df["mutation"].nunique()
        n_diseases = (
            screens_df["diseaseFromSource"].nunique()
            if "diseaseFromSource" in screens_df
            else None
        )

        context_tissues = [c.get("tissueName") for c in contexts if c.get("tissueName")]
        n_tissues = len(set(context_tissues))

        result = {
            "id": gene_id,
            # "n_contexts": n_contexts,
            "n_essential_contexts": n_essential_contexts,
            "essential_ratio": essential_ratio,
            "avg_gene_effect": avg_gene_effect,
            "std_gene_effect": std_gene_effect,
            "min_gene_effect": min_gene_effect,
            "max_gene_effect": max_gene_effect,
            "avg_expression": avg_expression,
            "max_expression": max_expression,
            "n_tissues": n_tissues,
            "n_mutations": n_mutations,
            "n_diseases": n_diseases,
        }

        # tissue-specific aggregations (context- and screen-level)
        for tissue in top_tissues:
            # context counts and essential ratios
            n_contexts_tissue = sum(
                1 for c in contexts if c.get("tissueName") == tissue
            )
            n_essential_tissue = sum(
                1
                for c in contexts
                if c.get("tissueName") == tissue and c.get("isEssential")
            )
            # result[f"context_count_{tissue}"] = n_contexts_tissue
            result[f"context_essential_ratio_{tissue}"] = (
                n_essential_tissue / n_contexts_tissue if n_contexts_tissue else None
            )

            # screen-level counts and averages by tissue
            tissue_screens = screens_df[screens_df["tissueName"] == tissue]
            # result[f"screen_count_{tissue}"] = len(tissue_screens)
            result[f"avg_gene_effect_{tissue}"] = (
                tissue_screens["geneEffect"].mean()
                if not tissue_screens.empty
                else None
            )

        result_records.append(result)
    result_records = pd.DataFrame(result_records)
    result_records = result_records.loc[:,result_records.nunique(dropna=False)!=1]
    return result_records


from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)

def run_catboost_cv(
    df,
    cat_features,
    text_features,
    embed_cols,
    cb_params,
    n_splits=5,
    group_col="targetId",
    merge_aba=False,
    random_state=42,
    return_model=True,  # model returned is a little different than not using pool..
    MINIMAL_FEATURES=True,
):
    """
    Single-run groupwise CV with CatBoost, splitting by group_col.

    Returns:
        oof_df  : DataFrame with columns ["pred", "label", group_col], aligned to df.index
        metrics_df : per-fold metrics
        summary    : dict with mean metrics across folds
        model      : last fitted CatBoost model (if return_model=True)
    """
    df = df.copy()
    s0 = df.shape[0]
    y_all = df["label"].astype(int)

    # Build features (may reorder/drop rows)
    X = get_cb_x(df, merge_aba=merge_aba,
                get_embeddings=False,
                add_target_feats=False,
                merge_mouse_pheno=False,
                merge_depmap_essentiality=False,
                get_network_embeds=False,
                embed_cols=embed_cols)
    assert s0 == X.shape[0], f"s0{s0} != X.shape[0]{X.shape[0]}"
    if MINIMAL_FEATURES:
        X = X.filter(['diseaseId', 'disease_text', 'target_text'],axis=1)
    # Keep only rows present in BOTH (preserve X's order for split indices)
    idx = X.index.intersection(df.index)
    X = X.loc[idx]
    assert s0 == X.shape[0], f"s0{s0} != X.shape[0]{X.shape[0]}; after index intersection"

    y = y_all.loc[idx]
    groups = df.loc[idx, group_col].values

    print(X.shape, "- X.shape")
    print(X.columns)

    gkf = StratifiedGroupKFold(
        n_splits=n_splits,
        random_state=random_state,
        shuffle=True,
    )

    # OOF over the *X* rows (positional)
    oof_pos = np.full(len(X), np.nan, dtype=float)
    fold_metrics = []
    last_model = None

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]

        train_pool = Pool(
            X_train, y_train,
            cat_features=cat_features,
            text_features=text_features,
            embedding_features=embed_cols,
        )
        val_pool = Pool(
            X_val, y_val,
            cat_features=cat_features,
            text_features=text_features,
            embedding_features=embed_cols,
        )

        model = CatBoostClassifier(**cb_params)
        model.fit(train_pool, verbose=200)  # optionally add eval_set / early stopping
        last_model = model

        preds = model.predict_proba(val_pool)[:, 1]
        oof_pos[val_idx] = preds  # positional write

        metrics = {
            "fold": fold,
            "roc_auc": roc_auc_score(y_val, preds),
            "prauc": average_precision_score(y_val, preds),
            "accuracy": accuracy_score(y_val, preds > 0.5),
            "f1": f1_score(y_val, preds > 0.5),
            "precision": precision_score(y_val, preds > 0.5),
            "recall": recall_score(y_val, preds > 0.5),
        }
        fold_metrics.append(metrics)

    metrics_df = pd.DataFrame(fold_metrics)
    summary = metrics_df.mean(numeric_only=True).round(4).to_dict()
    print(summary)

    # Expand OOF back to full df index (NaN where rows weren’t in X)
    # Compact OOF (only rows used by X)
    oof_series_compact = pd.Series(oof_pos, index=idx, name="pred")

    # Expand back to full df index, NaN where row wasn't in X (shouldn't happen, but safe)
    oof_series_full = pd.Series(np.nan, index=df.index, name="pred")
    oof_series_full.loc[idx] = oof_series_compact.values

    # ✅ Return the full original dataframe + OOF predictions
    # This preserves diseaseId, disease_text, source, etc.
    oof_df = df.copy()
    oof_df["pred"] = oof_series_full

    if return_model:
        return oof_df, metrics_df, summary, model
    else:
        return oof_df, metrics_df, summary


In [ ]:
def reset_pivoted_multiindex(df):
    df = df.sort_index(axis=1, level=1)
    df.columns = [f'{x}_{y}' for x,y in df.columns]
    df = df.reset_index()
    return df

##modified (index stuff):
def get_cb_x(df,
             keep_col=["diseaseId","disease_text","target_text"],
             drop_later_cols=["targetId"],
             get_embeddings=False,
             add_target_feats=True,
             merge_aba=False,
             merge_mouse_pheno=False,
             merge_depmap_essentiality=True,
             # get_network_embeds = None#True
             get_network_embeds = True
             ,embed_cols=None):  # <-- add this param explicitly (it was referenced)
    s_base = df.shape[0]
    df = df.copy()
    # carry a stable row id through merges
    df["_rid"] = df.index

    df = df.drop(columns=["label"], errors="ignore").filter(keep_col + drop_later_cols + ["_rid"], axis=1)

    if get_embeddings: 
        df = df.merge(df_embed_target, on="targetId", how="left")
        df = df.merge(df_embed_disease, on="diseaseId", how="left")

    if add_target_feats:
        if "id" in target_df.columns:
            target_df.rename(columns={"id":"targetId"}, inplace=True)
        df = df.merge(
            target_df.set_index("targetId")
                     .filter(target_df.select_dtypes(["number","bool"]).columns.tolist()
                             + ["subcellularLocations","biotype",], axis=1), # "tractability","go", already in text col  
            on="targetId", how="left"
        )
        df = df.merge(target_priority, on="targetId", how="left") # merge with target_priority features

    # disease-level extras
    df = df.merge(df_disease_pheno, on="diseaseId", how="left")
    df["diseaseHasPheno"] = df["diseaseId"].isin(df_disease_pheno["diseaseId"])

    if merge_mouse_pheno:
        df = df.merge(df_mouse_pheno_agg, on="targetId", how="left")
        df["TargetHasMousePheno"] = df["targetId"].isin(df_mouse_pheno_agg["targetId"])

    df = df.merge(
        disease_df[["diseaseId","ancestors",
                    # "therapeuticAreas","dbXRefs", ## already merged into in text col
                   ]],
        on="diseaseId", how="left"
    ).drop(columns=["id"], errors="ignore")

    if merge_depmap_essentiality:
        df = df.merge(target_essentiality, on="targetId", how="left")

    if merge_aba:
        s1 = df.shape[1]; s0 = df.shape[0]
        df = df.merge(df_aba.select_dtypes("number"), left_on="targetId", right_index=True, how="left")
        df["in_aba"] = df["targetId"].isin(df_aba.index).astype(int)
        print("in_aba\n", df["in_aba"].agg(["mean","sum","count"]).round(3))
        print(df.shape[1] - s1, "ABA columns added")
        assert s1 < df.shape[1]
        assert s0 == df.shape[0], f"rows mismatch from aba; premerge-aba: {s0} vs new: {df.shape[0]}"

    if get_network_embeds:
        df = df.merge(df_targ_network_embed, on="targetId", how="left")

    # clean strings (respect embed_cols if provided)
    for c in df.select_dtypes(["O","string"]).columns:
        if c == "_rid":
            continue
        if embed_cols is not None:
            if c not in embed_cols:
                df[c] = df[c].fillna("").astype(str)
        else:
            df[c] = df[c].fillna("").astype(str)

    # restore original index and drop helper cols
    assert s_base == df.shape[0], f"rows mismatch; expected {s_base}, got {df.shape[0]}"
    df = df.set_index("_rid").sort_index()
    df.drop(columns=drop_later_cols, errors="ignore", inplace=True)

    print(df.shape)
    return df

In [ ]:
# # 01_prepare_data.py
# # GOAL: To perform all heavy, one-time data processing and feature engineering.
# # This script is independent of the modeling library and produces clean, ML-ready files.
# ## run in advance 

class Config:
    DATA_DIR = "../data/opentargets/"
    PROCESSED_DATA_FILE = 'final_df.parquet'
    DISEASE_EMBEDDINGS_FILE = 'disease_embeddings.npz'
    TARGET_EMBEDDINGS_FILE = 'target_embeddings.npz'
    TEXT_MODEL = "nomic-ai/modernbert-embed-base"#'sentence-transformers/all-MiniLM-L6-v2'
# model = SentenceTransformer("nomic-ai/modernbert-embed-base")

In [ ]:
## note: gets updated later according to our df_int/learn targets data!
druggable_genome_list = pd.read_csv(os.path.join("../data", "finan_proc_druggable_genome_list.csv"))["ensembl_gene_id"].to_list()
print(len(druggable_genome_list))

## unique_targets 
unique_targets = druggable_genome_list
print(len(unique_targets),"# druggable genome targets")

* Target pipeline

In [ ]:
disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease')).filter(['id','name', 'description', 
                                                                               'dbXRefs',
                                                                               'synonyms',
                                                                               'ancestors',"parents",'therapeuticAreas'],axis=1).dropna(axis=1,how="all")
print(disease_df["id"].nunique(),"# unique Diseases")
# #ORIG
# # ## max 5 each. 
# disease_df["ExactSynonyms"] = disease_df["synonyms"].str.lower().apply(lambda d: list(dict.fromkeys(d.get("hasExactSynonym", [])))
#                         if isinstance(d, dict) else []) #PREV
disease_df["ExactSynonyms"] = disease_df["synonyms"].apply(lambda d: list(dict.fromkeys(d.get("hasExactSynonym", [])))[0:8]
                        )
disease_df["ExactSynonyms"] = disease_df["ExactSynonyms"].apply(" ".join).str.strip() # make str. 
# disease_df.drop(columns=["synonyms"],inplace=True,errors="ignore")

for c in disease_df.select_dtypes("O").columns:
    print(c)
    try:
        print(disease_df[c].str.len().describe().round(1),"len")
        ## truncate. Note: this is different for strings (description) vs lists! (# items)
        # Replaces: disease_df[c] = disease_df[c].str.slice(0,2_000)  # ORIG char based
        disease_df[c] = disease_df[c].apply(lambda x: " ".join(x.split()[:2_500]) if isinstance(x, str) else x[:1_000] if isinstance(x, list) else x)
    except:
        print("ERROR",c)

#Add joint text description col, used for embeddings.
## id is only relevant if using pretrained embeddings 
# 'id', 
# ##ORIG:
# disease_df["disease_text_embed"] = disease_df[['name',"ExactSynonyms","description"]].astype(str).apply(" ".join,axis=1)
##expanded text: 
disease_df["disease_text_embed"] = disease_df[['name',"ExactSynonyms","description",'dbXRefs','therapeuticAreas',"parents"]].astype(str).apply(" ".join,axis=1)
# disease_df.rename(columns={"id":"diseaseId"},inplace=True) # do it later ,for compat with target pipeline code
disease_df

In [ ]:
# disease_df.loc[disease_df["name"].str.contains("alzheimer",case=False)]
disease_df["disease_text_embed"].str.split().str.len().describe().round(1) # max 350

In [ ]:
disease_df.loc[disease_df["name"].str.contains("alzheimer",case=False)]["disease_text_embed"].iloc[0]

In [ ]:
print(disease_df["ExactSynonyms"].drop_duplicates().iloc[1])

### More disease - phenotype values
* count encoded
 * replace rare vals
  * non humanized values

In [ ]:
df_disease_pheno = pd.read_parquet(DATA_DIR+"disease_phenotype",columns=["disease","phenotype"]).rename(columns={"disease":"diseaseId"}).drop_duplicates()
# print(df_disease_pheno.shape[0])
# df_disease_pheno = df_disease_pheno.loc[df_disease_pheno["diseaseId"].isin(unique_diseases)]
print(df_disease_pheno.shape[0],"# rows after filtering by (valid) diseaseIds)")
print(df_disease_pheno.nunique())

#### count encoding
counts = df_disease_pheno['phenotype'].value_counts()
threshold = 3  # Example: phenotype/categories appearing 3 or less times are rare
rare_categories = counts[counts <= threshold].index
df_disease_pheno['phenotype'] = df_disease_pheno['phenotype'].replace(rare_categories, 'Rare').fillna("Unknown_Phenotypes")

# df_disease_pheno.groupby("phenotype").count().describe().round()

df_disease_pheno = df_disease_pheno.groupby("diseaseId").agg(phenotypes_count=('phenotype', 'count'),  # Get the count of 'Value' for each group
        phenotypes=('phenotype', lambda x: ' '.join(x)) 
    ).reset_index()
df_disease_pheno

In [ ]:
disease_df = disease_df.merge(df_disease_pheno[["diseaseId","phenotypes"]], left_on="id",right_on="diseaseId", how="left")
disease_df["phenotypes"] = disease_df["phenotypes"].fillna("Unknown_Phenotypes")
disease_df["disease_text_embed"] = (disease_df["disease_text_embed"]+" "+ disease_df["phenotypes"])
disease_df.drop(columns=["diseaseId","phenotypes"],inplace=True,errors="ignore") # we add diseaseId in later


In [ ]:
disease_df["disease_text_embed"].iloc[0]

## Target DF

In [ ]:
target_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'),columns=['id', 'approvedSymbol', 'biotype', #'transcriptIds', "canonicalExons",
       'genomicLocation', 'alternativeGenes', 'approvedName', 'go', 'hallmarks', 'synonyms',
       'functionDescriptions',  'subcellularLocations', 'targetClass', 'constraint', 'tep', 'proteinIds',# 'dbXrefs','chemicalProbes',
            # 'homologues', # useful and interesting but needs processing before use as feature - skip for now
         'tractability', 'safetyLiabilities',
       'pathways',# 'tss'
        ]).dropna(how="all",axis=1)#.rename(columns={"id":"targetId"}) # do rename later ,for compat with target pipeline code
print(target_df.shape[0])
# target_df = target_df.loc[target_df["targetId"].isin(valid_targets)]
print(target_df.shape[0])
assert target_df.shape[0]>0

# make into list of the go terms
target_df["go"] = target_df["go"].map(
    lambda x: list(dict.fromkeys(d["id"] for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)

target_df["proteinIds"] = target_df["proteinIds"].map(
    lambda x: list(dict.fromkeys(d["id"] for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)

target_df["synonyms"] = target_df["synonyms"].str.lower().map(
    lambda x: list(dict.fromkeys(d["label"] for d in x if isinstance(d, dict) and "label" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else [])
target_df["count_synonyms"] = target_df["synonyms"].str.len().fillna(0)
target_df["synonyms"] = target_df["synonyms"].str.slice(0,10) # first 10

target_df["count_subcellularLocations"] = target_df["subcellularLocations"].str.len().fillna(0) # before splitting
target_df["subcellularLocations"] = target_df["subcellularLocations"].map(
    lambda x:list(dict.fromkeys(d["location"] for d in x if isinstance(d, dict) and "location" in d)
             if isinstance(x, (list, tuple, np.ndarray)) else [])
)
# target_df["subcellularLocations"] = target_df["subcellularLocations"].str.join(", ")

# ## remove the "false" values, make it easier to extract
# target_df['tractability'] = target_df['tractability'].apply(
#     lambda v: list(dict.fromkeys([d for d in (v if isinstance(v, (list, tuple, np.ndarray)) else [v]]))
#                if isinstance(d, dict) and d.get('value', False))
# )
target_df["tractability"] = target_df["tractability"].apply(
    lambda v: list({                               # dedupe, keep insertion order (Py 3.7+)
        frozenset(d.items()): d                    # hashable surrogate key → original dict
        for d in (v if isinstance(v, (list, tuple, np.ndarray)) else [v])
        if isinstance(d, dict) and d.get("value")  # keep only truthy-value dicts
    }.values())
)
## get values from tractability. "value" seems to be only true or false. get vals for less text:

target_df["tractability"] = target_df["tractability"].map(
    lambda x: list((" ".join([d["modality"],d["id"]]) for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else [])

## make into string from list:
# target_df["go_terms_str"] = target_df["go_terms"].str.join(",")

## add some count features of lists of terms. NOTE: doesn't distinguish on values, or falses etc 
for c in ["go","pathways","proteinIds","targetClass","safetyLiabilities","tractability",
          "alternativeGenes","hallmarks","functionDescriptions","tep"]: # "homologues",
    if c in target_df.columns:
        target_df[f"count_{c}"] = target_df[c].str.len().fillna(0).astype(int)
        print(c,"max len",target_df[f"count_{c}"].max())
        try:
            ## truncate length! Also may ruin string format!
            # target_df[c] = target_df[c].str.slice(0,500) # ORIG char based
            target_df[c] = target_df[c].apply(lambda x: " ".join(x.split()[:2_000]) if isinstance(x, str) else x[:250] if isinstance(x, list) else x)
        except:
            print(c,"ERROR!")

target_df["functionDescriptions"] =  target_df["functionDescriptions"].map(lambda x: " ".join(x),na_action="ignore").fillna("").str.strip() # as str

target_df["synonyms"] = target_df["synonyms"].apply(" ".join).str.strip() # make str. 
#Add joint text description col, used for embeddings.
## 'id', 'approvedSymbol', - not use if using for featurees vs embed
## symbol without numbers - pseudo gene family
target_df["sym"] = target_df['approvedSymbol'].str.replace(r'\d+', '', regex=True)
# ##ORIG:
# target_df["target_text_embed"] = target_df[["sym","approvedName","synonyms","functionDescriptions"]].astype(str).apply(" ".join,axis=1)
##ALT: Expanded text
target_df["target_text_embed"] = target_df[["sym","approvedName","synonyms","functionDescriptions","go","tractability","pathways",
                                             'targetClass']].astype(str).apply(" ".join,axis=1) # , 'constraint' : numbers
def extract_constraint_scores(row, key):
    """
    Get constraint scores
    >
    target_df['score_syn'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'syn'))
    target_df['score_mis'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'mis'))
    target_df['score_lof'] = target_df['constraint'].apply(lambda x: extract_scores(x, 'lof'))
    """
    if isinstance(row, (list, np.ndarray)):
        for item in row:
            if isinstance(item, dict) and item.get('constraintType') == key:
                return item.get('score')
    return np.nan

# Apply to create new columns
target_df['score_syn'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'syn'))
target_df['score_mis'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'mis'))
target_df['score_lof'] = target_df['constraint'].apply(lambda x: extract_constraint_scores(x, 'lof'))

# target_df["hasSafetyLiabilities"] = target_df["safetyLiabilities"].isna().astype(int)
# target_df["Nohallmarks"] = target_df["hallmarks"].isna().astype(int)
print(target_df["id"].nunique(),"# unique targets")
display(target_df.head(3))

In [ ]:
# target_df["target_text_embed"].str.split().str.len().describe().round(1)

In [ ]:
target_df[target_df["tractability"].astype(str).str.contains("drug",case=False,na=False)]["id"].nunique()

In [ ]:
target_df[target_df["count_tractability"]>0]["id"].nunique()

In [ ]:
# target_df.drop_duplicates("hallmarks")["hallmarks"].values
target_df["hallmarks"].isna().mean() ## 99% nan/missing

In [ ]:
target_df["tractability"].head().values

#### mini filter of targets for tractability
* Could use 2017 list of around 2.5~4.5 K.
* For now, keep those with any tractability (no filter on quality) or known drug (1522) , leaving around 17K out of 78 K.
We will want the "Druggable genome" for candidates. (But not for Retrieval model training or targets)

In [ ]:
known_drug_ids = pd.read_parquet(os.path.join(Config.DATA_DIR, 'known_drug'),columns=["targetId"])["targetId"].unique() # 1522
print(target_df["id"].nunique()) 
target_df = target_df.loc[(target_df["count_tractability"]>0)|(target_df["id"].isin(known_drug_ids))]
print(target_df["id"].nunique(),"After tractability or known drug filterin")

In [ ]:
# known_drug_df["targetId"].nunique() # 1522

In [ ]:
target_df["target_text_embed"].str.split().str.len().describe().round(1)

In [ ]:
disease_df

In [ ]:
# 01_prepare_data.py
# GOAL: To perform all heavy, one-time data processing and feature engineering.
# This script is independent of the modeling library and produces clean, ML-ready files.
import os
import numpy as np
import pandas as pd

class Config:
    DATA_DIR = "../data/opentargets/"
    PROCESSED_DATA_FILE = 'final_df.parquet'
    DISEASE_EMBEDDINGS_FILE = 'disease_embeddings.npz'
    TARGET_EMBEDDINGS_FILE = 'target_embeddings.npz'
    # TEXT_MODEL = 'sentence-transformers/all-MiniLM-L12-v2' # originally used 
    TEXT_MODEL = 'thomas-sounack/BioClinical-ModernBERT-large'
    # TEXT_MODEL =  "nomic-ai/modernbert-embed-base" "lightonai/modernbert-embed-large" # Needs PREFIXES!! https://huggingface.co/lightonai/modernbert-embed-large 
    # https://huggingface.co/thomas-sounack/BioClinical-ModernBERT-base
# biolord; lokeshch19/ModernPubMedBERT 


### newly moved up here,outside the "main":
# ## Always run this section; so we can have its filtered diseases in data (for filtering candidate diseases):
### following section extracted; used to filter out diseases (e.g. "measurement")
ta_map = disease_df.set_index('id')['name'].to_dict()
labels_to_remove = ['measurement', 'phenotype', 'biological process', 'cell proliferation disorder']
exploded_tas = disease_df[['id', 'therapeuticAreas']].explode('therapeuticAreas').dropna()
exploded_tas['label'] = exploded_tas['therapeuticAreas'].map(ta_map)
ids_to_remove = exploded_tas[exploded_tas['label'].isin(labels_to_remove)]['id'].unique()

def make_target_data():
    """Orchestrates the data processing and feature generation pipeline."""
    print("--- Starting Data and Feature Pre-computation ---")
    
    if not os.path.exists(Config.PROCESSED_DATA_FILE):
        print(f"-> '{Config.PROCESSED_DATA_FILE}' not found. Processing raw data...")
        # disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease'))
        associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct'))
        known_drug_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'known_drug'))
        
        associations_filtered = associations_df[~associations_df['diseaseId'].isin(ids_to_remove)]
        ###### Should we be dropping phase - na cases? (dropna)?? (Also later on) ######
        validated_targets = known_drug_df.dropna(subset=['phase'])['targetId'].unique()
        working_df = associations_filtered[associations_filtered['targetId'].isin(validated_targets)].copy()
        
        pairs_with_evidence = known_drug_df.dropna(subset=['phase'])[['targetId', 'diseaseId']].drop_duplicates()
        pairs_with_evidence['label'] = 1
        
        final_df = pd.merge(working_df, pairs_with_evidence, on=['targetId', 'diseaseId'], how='left')
        final_df['label'] = final_df['label'].fillna(0).astype(int)
        print(final_df['label'].describe().round(3))
        final_df.to_parquet(Config.PROCESSED_DATA_FILE)
        print(f"-> Saved processed data to '{Config.PROCESSED_DATA_FILE}'.")
    else:
        print(f"-> Found '{Config.PROCESSED_DATA_FILE}'. Skipping raw data processing.")
def get_pretrained_text_embeddings():
    ## could filter by IDs in the training dataset? 
    #### note: normalize_embeddings may GREATLY reduce perf (from 96 to 90) ?
    if not os.path.exists(Config.DISEASE_EMBEDDINGS_FILE):
        from sentence_transformers import SentenceTransformer
        print("-> DISEASE Text embeddings not found. Generating now...")
        model = SentenceTransformer(Config.TEXT_MODEL, model_kwargs={"dtype": "bfloat16"})
        
        # disease_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'disease'))
        # target_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'))
        
        unique_diseases = disease_df[['id', 'disease_text_embed']].dropna().drop_duplicates('id').set_index('id')
        disease_embeddings = model.encode(unique_diseases['disease_text_embed'].tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False)
        # disease_embeddings = model.encode(("search_query: " + unique_diseases['disease_text_embed']).tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False) # if a model that needs prefixes
        np.savez_compressed(Config.DISEASE_EMBEDDINGS_FILE, ids=unique_diseases.index.values, embeddings=disease_embeddings)
    if not os.path.exists(Config.TARGET_EMBEDDINGS_FILE):
        from sentence_transformers import SentenceTransformer
        print("-> TARGET Text embeddings not found. Generating now...")
        model = SentenceTransformer(Config.TEXT_MODEL, model_kwargs={"dtype": "bfloat16"})
        # unique_targets = target_df[['id', 'approvedSymbol',"approvedName"]].dropna().drop_duplicates('id').set_index('id')
        # target_embeddings = model.encode((unique_targets['approvedSymbol']+" - "+unique_targets["approvedName"]).tolist(), show_progress_bar=True, device='cuda')
        unique_targets = target_df[['id', 'target_text_embed']].dropna().drop_duplicates('id').set_index('id')
        target_embeddings = model.encode((unique_targets['target_text_embed']).tolist(), show_progress_bar=True, device='cuda',normalize_embeddings=False,)
        # target_embeddings = model.encode((("search_document: " + unique_targets['target_text_embed']).tolist()), show_progress_bar=True, device='cuda',normalize_embeddings=False,) # if a model that needs prefixes
        np.savez_compressed(Config.TARGET_EMBEDDINGS_FILE, ids=unique_targets.index.values, embeddings=target_embeddings)
        print("-> Saved text embeddings.")
    else:
        print("-> Found text embeddings. Skipping generation.")
        
    print("\n--- Data and Feature Preparation Complete ---")

if __name__ == "__main__":
    make_target_data()
    get_pretrained_text_embeddings()

In [ ]:
try:
    tractability_all_targets_list = target_df[target_df["count_tractability"]>0]["id"].unique()
    tractability_drug_targets_list =target_df[target_df["tractability"].astype(str).str.contains("drug",case=False,na=False)]["id"].unique()
    
    print(len([c for c in validated_targets if c not in tractability_drug_targets_list])) # 275 , out of 1522
    print(len([c for c in validated_targets if c not in tractability_all_targets_list])) # 18 , out of 1522
except:()

In [ ]:
target_df.rename(columns={"id":"targetId"},inplace=True)
target_df["targetId"] = target_df.targetId.astype("category")
disease_df.rename(columns={"id":"diseaseId"},inplace=True)
disease_df["targetId"] = disease_df.diseaseId.astype("category")

In [ ]:
try: final_df.shape # see if variable not defined
except:
    final_df = pd.read_parquet(Config.PROCESSED_DATA_FILE)
df_int = final_df.drop(columns=["evidenceCount"],errors="ignore") # score may be nice proxy target, later
df_int.diseaseId = df_int.diseaseId.astype("category")
df_int.targetId = df_int.targetId.astype("category")
unique_diseases= df_int.diseaseId.unique()
# unique_targets = df_int.targetId.unique() # old
print(df_int.nunique())
print(df_int[["score","label"]].mean().round(3))
df_int

### join/union targets (druggable and our data)
* Druggable should theoretically include our data, but whatever, mismatch isnot' massive
* Note that we need to save this if using elsewhere

In [ ]:
print(len(unique_targets))
unique_targets = list(set(final_df.targetId.unique()) | set(unique_targets))
print(len(unique_targets))

assert len([c for c in df_int.targetId.astype(str) if c not in unique_targets])==0 
del final_df

In [ ]:
if FAST_RUN:
    df_int = df_int.sample(5_000).reset_index(drop=True)

In [ ]:
df_int.score.mean()

### OT Score metafeatures
* Features of the OT scores for candidates
* Some are leaks, e.g. if known drugs are known for target-disease
* We can use this so subset and find insights into our candidates/predictions.
    * e.g.: predictions or casess with low genetic/gwas evidence
    * e.g. etiology for treatable targets
    * ...

Features need to be subset. can also include indirect. 

In [ ]:
df_assoc_type = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_datatype_direct'))
print(df_assoc_type["datatypeId"].value_counts())
display(df_assoc_type)

df_assoc_direct = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct'))
df_assoc_direct["datatypeId"] = "direct" # pseudo val
display(df_assoc_direct)

In [ ]:
print("Distribution for known drugs:")
print("known drug - drug-evidence scores")
df_assoc_type_known_drug = df_assoc_type.query('datatypeId=="known_drug"')
print(df_assoc_type_known_drug.describe().round(2))
print("overall evidence scores - known drug (filtered)")
df_assoc_direct.merge(df_assoc_type_known_drug[["diseaseId","targetId"]],on=["diseaseId","targetId"],how="inner")["score"].describe().round(3)

In [ ]:
df_assoc_type.query('datatypeId=="known_drug"').sort_values("score").head()

In [ ]:
df_assoc_direct = pd.concat([df_assoc_direct,df_assoc_type.drop(columns=["evidenceCount"])])
# df_assoc_direct

In [ ]:
df_assoc_direct = reset_pivoted_multiindex(df_assoc_direct.pivot(index=["diseaseId","targetId"], columns='datatypeId',values=["score"]))
df_assoc_direct

## Extra features
* need to join +- process
* Easier to add in tree model section

In [ ]:
target_priority = pd.read_parquet(os.path.join(DATA_DIR, "target_prioritisation"))
print(target_priority.dropna(subset=["maxClinicalTrialPhase"])["targetId"].nunique())
## drop leaky columns: (+- tractabiltiy later)
target_priority = target_priority.drop(columns=["maxClinicalTrialPhase","hasHighQualityChemicalProbes"],errors="ignore").dropna(axis=0,thresh=2)
target_priority

#### target_essentiality
* depmap and gene essentiality. Run here due to it being slow - we filter it for our subset of targets first

parsing essentiality is slow + dozens of output cols! 

In [ ]:
%%time
# ## slow 
target_essentiality = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target_essentiality'))
print(target_essentiality.shape[0])
target_essentiality = target_essentiality.loc[target_essentiality["id"].isin(unique_targets)] # valid_targets
print(target_essentiality.shape[0])
# target_essentiality = preprocess_open_targets(target_essentiality)
target_essentiality = preprocess_open_targets(df=target_essentiality,top_n_tissues=20) # faster
print(target_essentiality.shape[1])
# target_essentiality = target_essentiality.loc[:,target_essentiality.nunique(dropna=False)!=1] # drop 0 var columns; added into function itself
target_essentiality.rename(columns={"id":"targetId"},inplace=True)
display(target_essentiality) # depmap

#### need to aggregate across rows - for mouse and disease phenotype datas

In [ ]:
%%time
df_mouse_pheno = pd.read_parquet(DATA_DIR+"mouse_phenotype",columns=['modelPhenotypeClasses', 'modelPhenotypeId',
       'modelPhenotypeLabel', 'targetFromSourceId', #'targetInModel', #  'targetInModelMgiId'
       'targetInModelEnsemblId',])
print(df_mouse_pheno.shape[0])
df_mouse_pheno = df_mouse_pheno.loc[df_mouse_pheno["targetFromSourceId"].isin(unique_targets)]
print(df_mouse_pheno.shape[0],"# rows after filtering by (valid) targetIDs)")
## high level phenotype
df_mouse_pheno["modelPhenotypeClasses"] = df_mouse_pheno["modelPhenotypeClasses"].map(
    lambda x: list(dict.fromkeys(d["label"].replace(" phenotype","") for d in x if isinstance(d, dict) and "id" in d))
             if isinstance(x, (list, tuple, np.ndarray)) else []
)
print(df_mouse_pheno.filter(['modelPhenotypeId', 'modelPhenotypeLabel', 'targetFromSourceId', 'targetInModel',
       'targetInModelEnsemblId', 'targetInModelMgiId']).nunique())
display(df_mouse_pheno)

#### count encoding
counts = df_mouse_pheno['modelPhenotypeId'].value_counts()
threshold = 3  # Example: phenotype/categories appearing 2 or less times are rare
rare_categories = counts[counts <= threshold].index
df_mouse_pheno['modelPhenotypeId'] = df_mouse_pheno['modelPhenotypeId'].replace(rare_categories, 'Rare')


### aggregate into 2 lists
print("Aggregated mouse phenos for targets:")
df_mouse_pheno_agg = df_mouse_pheno.groupby("targetFromSourceId")["modelPhenotypeClasses"].apply(lambda x: list(set([item for sublist in x for item in sublist])),include_groups=False).reset_index()
df_mouse_pheno_agg = df_mouse_pheno_agg.merge(df_mouse_pheno.groupby("targetFromSourceId")["modelPhenotypeId"].agg(lambda x: list(set(x))).reset_index(),
                                             on="targetFromSourceId",how="outer")
df_mouse_pheno_agg.rename(columns={"targetFromSourceId":"targetId"},inplace=True)
df_mouse_pheno_agg.drop_duplicates(subset=["targetId"],inplace=True)
display(df_mouse_pheno_agg)

## ABA - developing mouse brain
* map those genes to human genes (partial coverage) and add their features
* list of genes and their features extracted in `aba_download_genelist.ipynb`

In [ ]:
df_aba = pd.read_parquet("aba_DevMouseBrain_features.parquet")
# df_aba

In [ ]:
## homologues from target_df
df_mouseMap = pd.read_parquet(os.path.join(Config.DATA_DIR, 'target'),columns=['id', 'approvedSymbol','homologues']).dropna(how="all",axis=1)
df_mouseMap.rename(columns={"id":"targetId"},inplace=True)

def first_mouse_dict(v):
    seq = v if isinstance(v, (list, tuple, np.ndarray)) else [v]
    return next((d for d in seq if isinstance(d, dict) and d.get("speciesName") == "Mouse"), None)

df_mouseMap["mouse_homolog"] = df_mouseMap["homologues"].apply(first_mouse_dict)
df_mouseMap.dropna(subset=["mouse_homolog"],axis=0,inplace=True)

df_mouseMap["mouse_homolog_targetGeneId"] = df_mouseMap["mouse_homolog"].map(
    lambda d: d.get("targetGeneId") if isinstance(d, dict) else None
)
df_mouseMap["mouse_homolog_targetGeneSymbol"] = df_mouseMap["mouse_homolog"].map(
    lambda d: d.get("targetGeneSymbol") if isinstance(d, dict) else None
)
df_mouseMap.drop(columns=["homologues","mouse_homolog"],inplace=True, errors="ignore")
# df_mouseMap

In [ ]:
len(set(df_aba["gene_acronym"]).intersection(set(df_mouseMap["mouse_homolog_targetGeneSymbol"]))) # 1242

In [ ]:
s1 = df_aba.shape[0]
## mouse_homolog_targetGeneId
df_aba = df_aba.merge(df_mouseMap[["targetId","mouse_homolog_targetGeneSymbol"]],left_on="gene_acronym",right_on="mouse_homolog_targetGeneSymbol",how="inner")
print(s1 - df_aba.shape[0],"# aba genes without match")

## for convenience: set target as index + keep only numeric
df_aba = df_aba.set_index("targetId").select_dtypes("number")
display(df_aba)

### network, sequence embeddings from String
* preprocessed in `process_string_embed.ipynb` , saved to `"target_string_embed.parquet"`

In [ ]:
df_targ_network_embed = pd.read_parquet(os.path.join("../data/","target_string_embed.parquet"))
df_targ_network_embed

#### down sample (opt) data + subset feature coluns
* add more features later + autocast (keras featureSpace)

In [ ]:
disease_df = disease_df.drop_duplicates(subset=["diseaseId"])#.set_index("diseaseId")
target_df = target_df.drop_duplicates(subset=["targetId"])#.set_index("targetId")

In [ ]:
df_int

In [ ]:
df_learn = df_int#.sample(frac=0.33)#.sample(212_000)
# df_learn.drop(columns=["score"],errors="ignore",inplace=True)
df_learn.label.sum()

df_learn = df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")#,left_on="targetId",right_index=True)
df_learn.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")
train_df = df_learn
display(train_df)


# ---------------------------------------------------------
# 1)  make a **target–hold-out** split  (70 % / 30 %)
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

# unique targets → split the *IDs* (not the rows)
train_tids, test_tids = train_test_split(
    df_learn["targetId"].unique(),
    test_size=0.2,          # paper: 70 % targets train, 30 % test
    random_state=42,
    shuffle=True,
    stratify = df_learn.drop_duplicates(subset=["targetId"])["label"]
)

train_df = df_learn[df_learn["targetId"].isin(train_tids)].copy()
test_df  = df_learn[df_learn["targetId"].isin(test_tids )].copy()

# ---------------------------------------------------------
# 2)  optional: split a *validation* set from the train targets
#     (again on targetId, 10 % of the training targets)
# ---------------------------------------------------------
train_tids, val_tids = train_test_split(
    train_tids,
    test_size=0.05,
    random_state=42,
    shuffle=True,
)

val_df   = train_df[train_df["targetId"].isin(val_tids)].copy()
train_df = train_df[train_df["targetId"].isin(train_tids)].copy()

In [ ]:
print("train/val/test target %")
print(train_df["label"].mean().round(2))
print(val_df["label"].mean().round(2))
print(test_df["label"].mean().round(2))

# --- quick sanity checks : that targetIDs are disjoint between train, test, val sets -------------------
assert set(train_tids).isdisjoint(val_tids),  "train & val targets overlap"
assert set(train_tids).isdisjoint(test_tids), "train & test targets overlap"
assert set(val_tids  ).isdisjoint(test_tids), "val & test targets overlap"

# optional: every original row is in exactly one split
assert len(train_df) + len(val_df) + len(test_df) == len(df_learn), \
       "row counts don't add up – some rows were lost or duplicated"

In [ ]:
df_int.score.mean().round(3)

In [ ]:
## for use in init bias trick - faster convergence to class imbalance ? 
## https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#examine_the_class_label_imbalance
neg, pos = np.bincount(df_learn['label'])
initial_bias = np.log([pos/neg])
initial_bias

##### minimal sanity check
* Lecks learned id embeddings! 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import roc_auc_score

DO_LR_BASELINE = False 
if DO_LR_BASELINE:
    tfidf = TfidfVectorizer(max_features=2_000, min_df=4)
    Xtr   = tfidf.fit_transform(train_df["disease_text"] + " " + train_df["target_text"])
    Xval  = tfidf.transform(val_df["disease_text"] + " " + val_df ["target_text"])
    Xtest  = tfidf.transform(test_df["disease_text"] + " " + test_df ["target_text"])
    
    clf   = LogisticRegression(max_iter=100, n_jobs=-1).fit(Xtr, train_df["label"])
    print("VAL Sklearn ROC-AUC:", roc_auc_score(val_df["label"], clf.predict_proba(Xval)[:,1]))
    print("Test Skl ROC-AUC:", roc_auc_score(test_df["label"], clf.predict_proba(Xtest)[:,1]))
    
    del Xtr,Xval,Xtest,tfidf

In [ ]:
# ### make candidates
## WARNING! Currently missing candidates outsdie of the training data

# # keep exactly one row per target — required by FactorizedTopK
candidates_df = (
    df_learn[["targetId", "target_text"]]      # add other target-side cols later
        .drop_duplicates(subset=["targetId"])  # pandas helper 
        .reset_index(drop=True)
)

In [ ]:
if SAVE_INTERMEDIATES:
    ## save these for use in DL model +- catboost model
    ### training data subset - not all candidates!
    df_learn.to_parquet("../data/proc/df_learn.parquet",index=False)
    
    disease_df.to_parquet("../data/proc/disease_df.parquet",index=False)
    target_df.to_parquet("../data/proc/target_df.parquet",index=False)

## DL

* Alt rewrite model (still 2 tower): https://www.tensorflow.org/recommenders/examples/featurization#putting_it_all_together
* Moved and refactored to `1-Train-DL-Retriever.ipynb`

### extra eval on test set

In [ ]:
# # ════════════════════════════════════════════════════════════
# # 6.  Retrieval index
# # ════════════════════════════════════════════════════════════
# # cand_vec  = concat([k_fs({"text": candidates_df["target_text"]}),
# #                     targ_emb(targ_lookup(candidates_df["targetId"]))])
# if TRAIN_DL:
#     cand_vec  = concat([k_fs({"text": candidates_df["target_text"]})])
#     cand_embs = k_tower(cand_vec)
    
#     retrieval = BruteForceRetrieval(k=10, return_scores=True)
#     retrieval.update_candidates(cand_embs,
#                                 np.arange(len(candidates_df), dtype="int32"))
    
#     # Example predictions
#     qry_embed = model.encode_q(np.array(["lung fibrosis"]),
#                                np.array(["MONDO_0000160"]))
#     _, idx = retrieval(qry_embed)
#     print("Top‑10 targets:",
#           tf.gather(candidates_df["targetId"].to_numpy(), idx[0]).numpy().tolist())
#     ## reccommend and filter out known positives
#     # -------------------------------------------------------------------
#     # 1.  Build an int→string lookup tensor  (once)
#     # -------------------------------------------------------------------
#     tid_lookup = tf.constant(candidates_df["targetId"].to_numpy())  # shape (N,)
    
#     # ------------- create a set of known (diseaseId, targetId) positives
#     positives = set(zip(df_learn.query("label==1")["diseaseId"],
#                         df_learn.query("label==1")["targetId"]))
#     ######
#     def recommend(disease_id, disease_text, k=7, drop_known=True):
#         q = model.encode_q(np.array([disease_text]), np.array([disease_id]))
#         _, idx = retrieval(q)
#         cand = tf.gather(tid_lookup, idx[0]).numpy().astype(str)
#         if drop_known:
#             cand = [t for t in cand if (disease_id, t) not in positives]
#         return cand[:k]
    
#     # -------------------------------------------------------------------
#     # 2.  Pretty-print an example
#     # -------------------------------------------------------------------
#     targets_dict = target_df.reset_index().set_index("targetId")[["approvedSymbol","approvedName"]].to_dict(orient="index") # map IDs to readable names
    
#     example_id   = "MONDO_0000160"
#     example_text = "lung fibrosis" # needs actual disease text, nvm more features later
    
#     print("Top-10 *novel* targets for", example_text, ":")
#     # print("\n".join(recommend(example_id, example_text, drop_known=True)))
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])
#     print("\nMONDO_0100233 - long Covid:")
#     # res = recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")
#     example_id, example_text = disease_df.loc[disease_df["diseaseId"]=="MONDO_0100233"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     # print("\n".join(recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")))
#     print([*map(targets_dict.get, res)])

#     print("\n diabetes mellitus type 2 associated cataract")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="diabetes mellitus type 2 associated cataract"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])


#     ## Essential hypertension
#     print("\nEssential hypertension")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="essential hypertension"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])

#     print("\nMultiple sclerosis")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="multiple sclerosis"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])
    
#     print("\nAsperger syndrome/Autism")
#     example_id, example_text = disease_df.loc[disease_df["name"]=="Asperger syndrome"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])    

#     print("\ntreatment refractory schizophrenia") # Schizophrenia which does not respond to common
#     example_id, example_text = disease_df.loc[disease_df["name"]=="treatment refractory schizophrenia"][["diseaseId","disease_text_embed"]].iloc[0].values
#     res = recommend(example_id, example_text, drop_known=True)
#     print([*map(targets_dict.get, res)])    

## Evaluate baselines

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
)

def evaluate_prob_series(y_true, y_hat, name, threshold=0.5):
    # Handle epsilon vs eps for log_loss
    ll_kwargs = {"labels": [0, 1]}
    param_names = log_loss.__code__.co_varnames
    if "eps" in param_names:
        ll_kwargs["eps"] = 1e-7
    elif "epsilon" in param_names:
        ll_kwargs["epsilon"] = 1e-7

    # Binary predictions from scores
    y_pred_bin = (y_hat >= threshold).astype("int32")

    print(
        f"{name:18s}  "
        f"AUC={roc_auc_score(y_true, y_hat):.4f}  "
        f"PR-AUC={average_precision_score(y_true, y_hat):.4f}  "
        f"BCE={log_loss(y_true, y_hat, **ll_kwargs):.4f}  "
        f"Acc={accuracy_score(y_true, y_pred_bin):.4f}  "
        f"P={precision_score(y_true, y_pred_bin, zero_division=0):.4f}  "
        f"R={recall_score(y_true, y_pred_bin):.4f}"
    )

def target_mean_baseline(test_df, full_df):
    """Return target-wise oracle mean predictions from full dataset (df_learn)."""
    target_means = full_df.groupby("targetId")["label"].mean()
    return test_df["targetId"].map(target_means).fillna(target_means.mean()).to_numpy(dtype="float32")

y_test = test_df["label"].to_numpy(dtype="float32")

evaluate_prob_series(y_test,
                     test_df[["diseaseId"]].merge(train_df.groupby("diseaseId")["label"].mean(),on="diseaseId",how="left")["label"].fillna(0), # mean of disease in train, apply to test
                     "DISEASE mean")

### code for these baselines was deleted
# evaluate_prob_series(y_test,
#                      constant_baseline(test_df, train_df["label"].mean()),
#                      "GLOBAL")

# evaluate_prob_series(y_test,
#                      disease_mean_baseline(test_df, train_df),
#                      "DISEASE mean")

# evaluate_prob_series(
#     y_test,
#     target_mean_baseline(test_df, df_learn),
#     "TARGET mean (oracle)"
# )


## Make feature dataset for normal tabular learning


In [ ]:
df_learn[["diseaseId","targetId"]].nunique()

#### save intermediates

In [ ]:
if SAVE_INTERMEDIATES:
    # df_learn.to_parquet("./data_sb/merged_OT_v1.snappy.parquet",index=False,compression="snappy") # error: scala.NotImplementedError: Parquet file contains a LIST typed column[6], which is currently unsupported.
    df_learn.to_csv("../data/data_sb/merged_OT_v1.csv.gz",index=False,compression="gzip")
    target_df.loc[target_df["targetId"].isin(df_learn["targetId"])].to_csv("../data/data_sb/target_feats_v1.csv.gz",index=False,compression="gzip")
    
    # target_essentiality.loc[target_essentiality["id"].isin(df_learn["targetId"])].to_parquet("./data_sb/depmap25_v1.snappy.parquet",index=False,compression="snappy")

## OTP overall evidence score baseline

* Our model outperforms the (train on train) baseline of the OT direct evidence score also!
* ~94 auc (cv) vs 91.36 rocauc from the OT score
* PR-AUC: 0.4546

In [ ]:
print(df_int.shape[0])
print("OTP Score ROCAUC:", roc_auc_score(df_int["label"], df_int["score"]))
print("OTP Score PR-AUC:", average_precision_score(df_int["label"], df_int["score"]))
## we don't have a cutoff for the score to make it classification trivially
# print("OTP Score classification:\n", classification_report(df["label"], df["score"]))


## Load Precomputed text embeddings


In [ ]:
import numpy as np
import pandas as pd
# Load the .npy file using numpy.load()
npz = np.load('disease_embeddings.npz',allow_pickle=True)
# Convert the loaded NumPy array to a Pandas DataFrame
df_embed_disease= pd.DataFrame.from_dict({item: npz[item] for item in npz.files}, orient='index').T
df_embed_disease.rename(columns={"ids":"diseaseId","embeddings":"diseaseEmbeddings"},inplace=True)
# df_embed_disease = df_embed_disease.loc[df_embed_disease["diseaseId"].isin(df_learn["diseaseId"])]

# s1 = df_learn.shape[0]
# df_learn = df_learn.merge(df_embed_disease,on="diseaseId",how="left")
# assert s1 == df_learn.shape[0]

npz = np.load('target_embeddings.npz',allow_pickle=True)
# Convert the loaded NumPy array to a Pandas DataFrame
df_embed_target= pd.DataFrame.from_dict({item: npz[item] for item in npz.files}, orient='index').T
df_embed_target.rename(columns={"ids":"targetId","embeddings":"targetEmbeddings"},inplace=True)

# s1 = df_learn.shape[0]
# df_learn = df_learn.merge(df_embed_target,on="targetId",how="left")
# assert s1 == df_learn.shape[0]

In [ ]:
df_learn.diseaseId.nunique()

## Local Catboost/Tree model
* Can use embeddings (pretrained) , and get basic text and categorical features
* https://catboost.ai/docs/en/concepts/python-usages-examples#class-with-array-like-data-with-numerical,-categorical-and-embedding-features

* https://chatgpt.com/share/688fd1c2-569c-8013-92ec-d11748791a5c

## Tree model
#### very compute intensive! may hang and takes a while! 
* not sure what is heaviest part
* `embedding_features` (when explicitly fed a such to CB) are very _heavy_!


* text features in catboost example :
    * https://colab.research.google.com/github/catboost/tutorials/blob/master/text_features/text_features_in_catboost.ipynb#scrollTo=4I1cDLrr4T-b
    * CB also supports BPE, dictionaries= ['Word:min_token_occurrence=5']...

In [ ]:
%%time
y_train = train_df.label
y_test = test_df.label
y_val = val_df.label

X_train = get_cb_x(train_df)
X_test = get_cb_x(test_df)
X_val = get_cb_x(val_df)

In [ ]:
X_train.select_dtypes(["number","bool"]).columns.tolist()[0:40]

In [ ]:
text_features = [c for c in text_features if c in X_test.columns]
cat_features = [c for c in cat_features if c in X_test.columns]
if embed_cols: #not none
    embed_cols = [c for c in embed_cols if c in X_test.columns]

In [ ]:
## if want to add (catboost supported) embedding features (pretrained, target, disease)
# X_train_emb['embeddings'] = X_train.values.tolist() 
# and set embed cols


In [ ]:

import gc
try:
    del model # clean dl model
except:()
gc.collect()

#### Train CB model

In [ ]:
learn_pool = Pool(
    X_train, 
    y_train, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)

val_pool = Pool(
    X_val, 
    y_val, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)

test_pool = Pool(
    X_test, 
    y_test, 
    cat_features=cat_features,
    text_features=text_features,
    embedding_features=embed_cols,
)
print("Pools ready")
# model = CatBoostClassifier(**CFG["cb_params"])
model = CatBoostClassifier(iterations=40 if FAST_RUN else 300, depth=5 if FAST_RUN else 8,
                          eval_metric="AUC", early_stopping_rounds=50, 
                          task_type="GPU", ## runs out of mem with many cols
                          random_seed=42,auto_class_weights = "SqrtBalanced",)
model.fit(learn_pool, eval_set=val_pool, verbose=False,plot=True) # , verbose=100

In [ ]:
test_preds = model.predict(test_pool,prediction_type="Probability")[:,1]
print("rocauc:", round(roc_auc_score(y_true=y_test,y_score=test_preds),4))
print("average prec (pseudo pr-auc?):", round(average_precision_score(y_true=y_test,y_score=test_preds),4))
print(classification_report(y_true=y_test, y_pred=model.predict(test_pool)))

```
rocauc: 0.9398
average prec (pseudo pr-auc?): 0.7573
              precision    recall  f1-score   support

           0       0.96      0.97      0.96    116258
           1       0.70      0.64      0.67     14466

    accuracy                           0.93    130724
   macro avg       0.83      0.80      0.81    130724
weighted avg       0.93      0.93      0.93    130724
```

In [ ]:
model.get_feature_importance(data=test_pool,
                       # reference_data=None,
                       # type=EFstrType.FeatureImportance,
                       prettified=True).head(20).round(2)

In [ ]:
model.get_feature_importance(data=val_pool,
                       # reference_data=test_pool,
                       # type="ShapValues",
                       type="SageValues",
                       sage_n_samples=3_048,
                       prettified=True).head(15).round(3)

#### try to get shap interactions - Slowish

In [ ]:
# ## this is interactions of pairs, including linear contributions. 
# pd.DataFrame(model.get_feature_importance(val_pool, type='Interaction'),
#              columns=["i","j","score"]) \
#   .assign(feat_i=lambda d: d["i"].map(dict(enumerate(X_val.columns))),
#           feat_j=lambda d: d["j"].map(dict(enumerate(X_val.columns)))) \
#   [["feat_i","feat_j","score"]] \
#   .sort_values("score", ascending=False) \
#   .head(10).round(2)

* Get the interaction contribution (diagonal on shap), beyond just how much each added individually

In [ ]:
%%time
import numpy as np, pandas as pd, shap
## gets stuck on full data! / slow
def top_shap_interactions(model, val_pool, feature_names, top_k=10):
    explainer = shap.TreeExplainer(model)
    S = explainer.shap_interaction_values(val_pool)  # (n, m, m) or (n, m, m, C)

    # Handle multi-class by averaging over classes
    if isinstance(S, np.ndarray) and S.ndim == 4:
        S = S.mean(axis=-1)  # average over classes -> (n, m, m)

    # Global mean |interaction| for each pair
    S_abs = np.abs(S).mean(axis=0)  # (m, m)
    main_abs = np.abs(np.diagonal(S, axis1=1, axis2=2)).mean(axis=0)  # (m,)
    np.fill_diagonal(S_abs, 0.0)

    m = len(feature_names)
    rows = []
    for i in range(m):
        for j in range(i+1, m):
            inter = S_abs[i, j]
            denom = (main_abs[i] + main_abs[j]) + 1e-12
            rows.append((feature_names[i], feature_names[j], inter, inter/denom))

    return (pd.DataFrame(rows, columns=["feat_i","feat_j","interaction_abs_mean","synergy_ratio"])
              .sort_values(["interaction_abs_mean","synergy_ratio"], ascending=False)
              .head(top_k)
              .round(3))

# # # # usage:
# ###  Can run out of memory!! (But useful analysis!)
# top_synergy = top_shap_interactions(model, val_pool, list(X_val.columns), top_k=15)
# top_synergy

 try to check intersection of go terms
* may need to harmonize acronym/text
* no matching go terms, assuming format is same (not sure)

No intersections found , naively

In [ ]:
del learn_pool,val_pool,test_pool
del X_train,X_test,X_val

## CV Results
* run CV
* optional: analysis with and without some input ?

```
count    12337.00
mean         5.47
std         26.21
min          0.00
25%          0.00
75%          0.00
max        628.00
Name: num_known_targets, dtype: float64
```

## added repeated cv code:

In [ ]:
def run_catboost_repeated_cv(
    df,
    cat_features,
    text_features,
    embed_cols,
    cb_params,
    n_splits=5,
    n_repeats=5,
    group_col="targetId",
    merge_aba=True,
    base_random_state=42,
    return_model=True,
):
    """
    Repeated groupwise CV (e.g. 5x5) with CatBoost.

    - Uses StratifiedGroupKFold splits by `group_col` so targets are disjoint
      between train/val in every fold.
    - Repeats the CV `n_repeats` times with different random seeds.
    - Returns OOF predictions averaged across repeats (per row).

    Returns:
        oof_df      : DataFrame ["pred","label",group_col], pred = mean OOF over repeats
        metrics_df  : per-fold + per-repeat metrics (cols: repeat, fold, roc_auc, prauc, ...)
        summary     : mean of metrics across all folds and repeats
        model       : last fitted model (if return_model=True)
    """
    df = df.copy()
    index_template = df.index.copy()

    all_oof = []        # list of np.array, one per repeat
    all_metrics = []    # list of metrics_df, one per repeat
    last_model = None
    ##added
    text_features = [c for c in text_features if c in df.columns]
    cat_features = [c for c in cat_features if c in df.columns]
    if embed_cols: #not none
        embed_cols = [c for c in embed_cols if c in df.columns]
    
    for rep in range(n_repeats):
        seed = None if base_random_state is None else base_random_state + rep
        print(f"\n=== Repeated CV run {rep+1}/{n_repeats} (seed={seed}) ===")

        oof_df_single, metrics_df_single, summary_single, model_single = run_catboost_cv(
            df=df,
            cat_features=cat_features,
            text_features=text_features,
            embed_cols=embed_cols,
            cb_params=cb_params,
            n_splits=n_splits,
            group_col=group_col,
            merge_aba=merge_aba,
            random_state=seed,
            return_model=True,
        )

        # Ensure OOF preds are aligned to the original df order
        preds_aligned = oof_df_single.loc[index_template, "pred"].to_numpy()
        all_oof.append(preds_aligned)

        metrics_df_single = metrics_df_single.copy()
        metrics_df_single["repeat"] = rep
        all_metrics.append(metrics_df_single)

        last_model = model_single

    # Shape: (n_repeats, n_samples)
    all_oof = np.vstack(all_oof)
    mean_oof = np.nanmean(all_oof, axis=0)  # just in case there were NaNs

    # Build final OOF df: same structure as before, but preds averaged across repeats
    # Build final OOF df: original df + averaged predictions
    oof_df = df.copy()
    oof_df["pred"] = mean_oof

    metrics_df = pd.concat(all_metrics, ignore_index=True)
    summary = metrics_df.mean(numeric_only=True).round(4).to_dict()
    print("Repeated-CV summary:", summary)

    if return_model:
        return oof_df, metrics_df, summary, last_model
    else:
        return oof_df, metrics_df, summary


In [ ]:
# train_df[["diseaseId","targetId"]+text_features]

In [ ]:
train_df.columns

* new run without added features

In [ ]:
%%time
if RUN_CV:
    df_all = pd.concat([train_df, test_df, val_df]).drop_duplicates()#.sample(n=1_000)
    
    oof_pred, fold_metrics, summary, model = run_catboost_repeated_cv(
        df=df_all,
        cat_features=cat_features,
        # text_features=text_features,
        # embed_cols=embed_cols,
        # cat_features=[],
        text_features=text_features,
        embed_cols=embed_cols,
        cb_params=CFG["cb_params"],
        n_splits=5,
        n_repeats=5,          # 5×5 CV
        group_col="targetId", # target-disjoint folds
        merge_aba=False,      # or True, same as you used before
    )
    
    print("Mean metrics summary over 5×5 CV:")
    print({k: round(v, 4) for k, v in summary.items()})
    
    # sanity checks (same as you had)
    assert oof_pred.dropna(axis=0).shape[0] == oof_pred.shape[0]
    assert oof_pred.shape[0] == df_all.shape[0]

In [ ]:
if RUN_CV:
    display(oof_pred)
    display(fold_metrics)
    print("SD:")
    print(fold_metrics.std().round(5))
    print("Summary")
    display(summary)

    oof_pred.to_parquet("./Outputs/CV_tree/CB_5_cv.parquet")
    fold_metrics.to_csv("./Outputs/CV_tree/CB_5_cv_foldMetrics.csv",index=False)
    # summary.to_csv("./Outputs/CV_tree/CB_5_cv_summary.csv",index=False)

```
SD:

roc_auc      0.00604
prauc        0.02186
accuracy     0.00324
f1           0.02046
precision    0.01856
recall       0.03059
 ```
fold_metrics
 ```
{ 'roc_auc': 0.9581,
 'prauc': 0.8203,
 'accuracy': 0.9474,
 'f1': 0.736,
 'precision': 0.7526,
 'recall': 0.7207,
 }
 ```

### Orig cv code follows: 

In [ ]:
# %%time
# df_all = pd.concat([train_df,test_df,val_df]).drop_duplicates()
# oof_pred, fold_metrics, summary,model = run_catboost_cv(
#     df=df_all,
#     cat_features=cat_features,         # e.g. ["diseaseId","biotype"]
# a    text_features=text_features,       # your long list with text columns
#     embed_cols=embed_cols,             # None unless you’re using embeddings
#     cb_params=CFG["cb_params"],        # your CatBoost params dict
#     n_splits=2 if FAST_RUN else 5,                        # or CFG["cv_folds"]
#     group_col="targetId",              # disjoint targets between train/val
#     merge_aba=False, #True
# )

# # print("Per-fold metrics:")
# # display(fold_metrics.round(4))
# print("Mean metrics summary:")
# print({k: round(v, 4) for k, v in summary.items()})

# assert oof_pred.dropna(axis=0).shape[0]==oof_pred.shape[0]
# assert oof_pred.dropna(axis=0).shape[0]==df_all.shape[0]



In [ ]:
df = df_all.copy()
df["num_known_targets"] = df.groupby("diseaseId")["label"].transform("sum") # for sorting by cases without known
df["in_aba"] = df["targetId"].isin(df_aba.index).astype(int)
df["pred"] = oof_pred["pred"]
display(df.groupby(["in_aba"])[["pred","label"]].mean().round(3))
df["pred_label"] = (df["pred"]>0.502).astype(int)
df["mistake"] = df["pred_label"]!=df["label"]
display(df["mistake"].agg(["mean","sum"]).round(2)) ## this accuracy seems wrong! maybe proba needs weighting? or get actual preds? 

Mean metrics summary:
{'fold': 2.0, 'roc_auc': 0.9432, 'prauc': 0.76, 'accuracy': 0.9409, 'f1': 0.6774, 'precision': 0.7545, 'recall': 0.6158}
### before adding in targ string embeds

#### add in sorting by diseases with least number of known positives

In [ ]:
df_cands = df.round(2).query('mistake & label==0 & pred>0.51').sort_values(["pred","score"],ascending=False)#.head(15)
df_cands.head(20)

In [ ]:
df.round(2).query('num_known_targets<2 & label==0 & pred>0.51 ').sort_values(["pred","score","diseaseId"],ascending=False)[['diseaseId', 'targetId', 'disease_text',
       'target_text', 'num_known_targets',  'pred', 'pred_label']]#.to_csv("orphan_candidates_v1.csv",index=False)

In [ ]:
# df_cands.head(4000).to_csv("sample_novel_preds_v1.csv")

In [ ]:
# df.loc[df["disease_text"].str.contains("Multiple sclerosis",case=False)] ## 253 targets
df.loc[(df["disease_text"].str.contains("multiple sclerosis")) & (df["mistake"]) & (df["pred_label"]>0)].sort_values(["pred","score","diseaseId"],ascending=False).round(2)

In [ ]:
df.query('mistake & label>0').sort_values("pred").round(1).head(10)

In [ ]:
df.query('mistake & label==0').sort_values(["score","pred"],ascending=False).round(1).head(10)

In [ ]:
fold_metrics.mean().round(3)

In [ ]:
fold_metrics.std().round(3)

In [ ]:
oof_df = oof_pred
# Evaluate only on rows that actually received OOF preds
mask = oof_df["pred"].notna()
y_oof = oof_df.loc[mask, "label"].values
p_oof = oof_df.loc[mask, "pred"].values
pred_lbl = (p_oof > 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
overall = {
    "accuracy": accuracy_score(y_oof, pred_lbl),
    "roc_auc": roc_auc_score(y_oof, p_oof),
    "prauc": average_precision_score(y_oof, p_oof),
    "f1": f1_score(y_oof, pred_lbl),
    "precision": precision_score(y_oof, pred_lbl),
    "recall": recall_score(y_oof, pred_lbl),
}
overall

In [ ]:
df_assoc_direct.columns

In [ ]:
df.loc[df[["pred","label"]].max(axis=1)>0.5]

In [ ]:
df2 = df.merge(df_assoc_direct,on=["diseaseId","targetId"])

# ## Keep "positve" cases: real or predicted
# df2 = df2.loc[df2[['label', 'pred_label']].max(axis=1)>0].round(4)
df2

In [ ]:
print("corr - in positives interactions (real or predicted)")
df2.select_dtypes(["number","bool"]).corr().round(2).loc[['score', 'label', 'pred', 'pred_label',
       'mistake',  'score_animal_model',
       'score_direct', 'score_genetic_association', 'score_known_drug',]]

In [ ]:
df2.select_dtypes(["number","bool"]).groupby("label").mean().round(2)

### Add all druggable genome X diseases as additional candidates data
* Maybe filter for subset of diseases due to size!
* Need to filter for cases not already done
* We'll get predictions from prev.trained model on these.
* Keep all druggable genome targets (not already covered) togethjer with diseases.

* will need to add score from: associations_df 

In [ ]:
# df_learn = df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
# df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")#,left_on="targetId",right_index=True)
# df_learn.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")


# disease_df
# target_df

In [ ]:
target_df_druggable = target_df.query("targetId in @druggable_genome_list").reset_index(drop=True)
print(target_df_druggable.shape[0])
## drop cases already covered - assuming any in relationship (and not "target+disease"):
## should drop around 1300~
target_df_druggable = target_df_druggable.query("targetId not in @df_all.targetId").reset_index(drop=True)
print(target_df_druggable.shape[0]) # 3080

####  Warning! We are taking only cases with any sort of evidence/score for the pair - we'll be missing cases with 0 evidence at all. 
* That's still most!
* We could use indirect evidence OR simply get "all to all" merge/join!!
* 
* This is IMPORTANT ! 

In [ ]:
# #### NOTE! These are only cases with some sort of direct evidence! Others will be missing: fill as 0.  (+- look at indirect)
# associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_overall_direct')).query("targetId in @druggable_genome_list")
# associations_df # 1.6M direct candidates

### done naively, this is a huge number of candidates! 120 M !
* We will take as candidates those with any indirect association , to speed thigns up : ~ 4M candidates! 

In [ ]:
indir_associations_df = pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_overall_indirect'),columns=['diseaseId', 'targetId', 'score']).query("targetId in @druggable_genome_list")
print(indir_associations_df.shape[0])
indir_associations_df = indir_associations_df[~indir_associations_df['diseaseId'].isin(ids_to_remove)] # drop some unwanted diseases, like cell proliferation, measurement
print(indir_associations_df.shape[0])
indir_associations_df

if FAST_RUN:
    indir_associations_df = indir_associations_df.head(n=80_000)
# indirect associations: more liberal. Can use as starting point.
## Still need score from direct associations score df (and fillna as 0)

In [ ]:
indir_associations_df.drop_duplicates(subset=['diseaseId', 'targetId']).shape

In [ ]:
# --- 2. The anti-join by 2 keys One-Liner ---
# We use a left merge with an indicator.
# Rows from 'df' that find a match in 'indir_associations_df' will be marked as 'both'.
# Rows from 'df' that DO NOT find a match will be marked as 'left_only'.
# We then query for 'left_only' (the rows we want to keep) and drop the helper column.
# We also drop_duplicates from the exclusion frame for efficiency.

df_cands_novel = indir_associations_df.merge(
    df_all[['diseaseId', 'targetId']].drop_duplicates(),
    on=['diseaseId', 'targetId'],
    how='left',
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')
df_cands_novel

In [ ]:
assert df_cands_novel.drop_duplicates(subset=['diseaseId', 'targetId']).shape[0] == df_cands_novel.shape[0]

In [ ]:
# %%time
# df_cands_novel = target_df_druggable[["targetId","target_text_embed"]].merge(disease_df[["diseaseId","disease_text_embed"]],how="cross") # 120M ! huge! 

In [ ]:
# disease_df[["diseaseId","disease_text_embed","name"]].nunique() # 38K

# df_learn.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")#,left_on="diseaseId",right_index=True)
# df_learn = df_learn.merge(target_df[["targetId","target_text_embed"]],on="targetId")

###### TODO: Why are there different # cnadiadtes?

In [ ]:
s1 = df_cands_novel.shape[0]
print(s1,"# cands pre merge")
df_cands_novel = df_cands_novel.merge(disease_df[["diseaseId","disease_text_embed"]],on="diseaseId")
df_cands_novel = df_cands_novel.merge(target_df[["targetId","target_text_embed"]],on="targetId")
df_cands_novel.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")
df_cands_novel

if s1 != df_cands_novel.shape[0]:
    print("Warning: # candidates changed")
    print(s1 - df_cands_novel.shape[0])

In [ ]:
import numpy as np
import pandas as pd
import gc

def predict_in_chunks(
    df: pd.DataFrame,
    model,
    feature_fn,
    chunk_size: int = 250_000,
    feature_kwargs: dict | None = None
) -> pd.DataFrame:
    """
    Lazily feature-ify and score df in chunks, then return df with a new 'pred' column.
    Assumes model.predict(..., prediction_type='Probability') returns either shape (n,2) or (n,).
    """
    feature_kwargs = feature_kwargs or {}

    n = len(df)
    preds = np.full(n, np.nan, dtype=float)

    # map original row index -> absolute position for stable write-back
    pos = pd.Series(np.arange(n), index=df.index, copy=False)

    start = 0
    while start < n:
        end = min(start + chunk_size, n)

        # slice without copying columns unnecessarily
        chunk = df.iloc[start:end]

        # build features for this chunk
        X = feature_fn(chunk, **feature_kwargs)
        display(X.iloc[:,15:19])
        # display(X.iloc[:,10:19].dtypes)
        # score; handle (n,2) vs (n,) outputs
        out = model.predict(X, prediction_type="Probability")
        proba = out[:, 1] if getattr(out, "ndim", 1) == 2 else np.asarray(out, dtype=float)

        # align-by-index in case feature_fn reorders rows
        idx_positions = pos.loc[X.index].to_numpy()
        preds[idx_positions] = proba

        # free memory
        del X, out, proba, idx_positions, chunk
        gc.collect()

        start = end

    # attach predictions to a shallow copy of the original frame
    result = df.copy()
    result["pred"] = preds
    return result

In [ ]:
%%time
# choose the same flags you already use for training/inference
feature_kwargs = dict(
    # get_embeddings=False,
    # add_target_feats=True,
    merge_aba=False,
    # merge_mouse_pheno=False,
    # merge_depmap_essentiality=True,
    # get_network_embeds=True,
    # embed_cols=None,
)


df_scored = predict_in_chunks(
    df=df_cands_novel,
    model=model,
    # model = CatBoostClassifier(**cb_params),
    feature_fn=get_cb_x,
    chunk_size=400_000,
    feature_kwargs=feature_kwargs
)

##### Add (indirect?) association scores, by source, as metadata
* could use direct or indirect
* total score and/or by source (note by known drug)

* Or use the (already loaded) direct sources, assuming it has coverage (and wasn't filtered)?

In [ ]:
# df_assoc_indir = reset_pivoted_multiindex(pd.read_parquet(os.path.join(Config.DATA_DIR, 'association_by_datatype_ןמdirect'),columns=['diseaseId', 'targetId', 'score']).pivot(index=["diseaseId","targetId"], columns='datatypeId',values=["score"]))
# df_assoc_indir

In [ ]:
# 1. Group df and sum 'label'
num_known_targets_counts = df.groupby("diseaseId")["label"].sum()
# 2. Map the sums to df_scored using 'diseaseId' and fill NaNs with 0
df_scored["num_known_targets"] = df_scored["diseaseId"].map(num_known_targets_counts).fillna(0)
# df_scored["num_known_targets"] = df.groupby("diseaseId")["label"].transform("sum") # for sorting by cases without known
df_scored["in_aba"] = df_scored["targetId"].isin(df_aba.index).astype(int)
df_scored = df_scored.merge(df_assoc_direct,on=["diseaseId","targetId"],how="left").dropna(axis=1,thresh=100)
df_scored["pred_label"] = (df_scored["pred"]>0.5).astype(int)

In [ ]:
df_scored

In [ ]:
df2.select_dtypes("number").mean().round(3)

In [ ]:
df_scored.select_dtypes("number").mean().round(3)

##### Q: We see "novel" candidates with possible associations - reasonable?
* these are from indirect assoc; but score is from direct
* maybe drug not same as clinicial trial! (our real criteria for default label)

In [ ]:
print(df2.shape[0])
# _1 = df2.query("(label>0)|(pred_label>0)").dropna(axis=1,thresh=50).round(3).sort_values(["label","pred","score","diseaseId"],ascending=[True,False,True,True]).drop(columns=["mistake"],errors="ignore")
# print(_1.shape)
print(df_scored.shape[0])
# _2 = df_scored.query("pred_label>0").dropna(axis=1,thresh=50).round(3).sort_values(["pred","score","diseaseId"],ascending=[False,True,True])
# print(_2.shape)

# _1.to_csv("positive_preds_trainData.csv")
# _2.to_csv("positive_preds_NovelIndirData.csv")

df_novel_pos_predictions = pd.concat([df2.query("(label>0)|(pred_label>0)").dropna(axis=1,thresh=50).round(3).sort_values(["label","pred","score","diseaseId"],ascending=[True,False,True,True]).drop(columns=["mistake"],errors="ignore"),
                                 df_scored.query("pred_label>0").dropna(axis=1,thresh=50).round(3).sort_values(["pred","score","diseaseId"],ascending=[False,True,True])],ignore_index=True)
df_novel_pos_predictions.drop(columns=['disease_text', 'target_text'],inplace=True,errors="ignore") # drop long terxt descriptions for final output - save on space
df_novel_pos_predictions.drop_duplicates(subset=["diseaseId","targetId"],inplace=True)
print(df_novel_pos_predictions.shape)
df_novel_pos_predictions = df_novel_pos_predictions.merge(disease_df[['diseaseId',"name"]].rename(columns={"name":"disease_name"}),on="diseaseId",how="left")
df_novel_pos_predictions = df_novel_pos_predictions.merge(target_df[['targetId', 'approvedSymbol', 'biotype', 'approvedName']].rename(columns={"approvedName":"Gene_name"}),on="targetId",how="left")
print(df_novel_pos_predictions.shape[0])
## reorder columns
df_novel_pos_predictions = df_novel_pos_predictions[[ 'disease_name', 'approvedSymbol', 
   'Gene_name', 'label',  'pred', 'pred_label', 'diseaseId', 'targetId', 'num_known_targets',
   'in_aba','score_direct', 'score_known_drug', #  'score',
   'score_literature', 'score_affected_pathway',
   'score_genetic_association', 'score_rna_expression',
   'score_somatic_mutation','num_known_targets',
   'biotype','in_aba',]].sort_values(['label',"pred","score_direct","diseaseId"],ascending=[True,False,False,True])
display(df_novel_pos_predictions)

In [ ]:
# if SAVE_NOVEL_PREDICTION_CANDIDATES:
#     df_novel_pos_predictions.to_csv("positive_preds_train_and_novelIndirect.csv",index=False)

In [ ]:
df_novel_pos_predictions.dropna(subset=["score_direct"],axis=0)